# Windows, timestamps and what is available at issue time

**Optional code walkthrough for AI 101 (Module 12).** Not graded. Quiz scores do not depend on it.

**Objective.** Check the 96-samples-per-day rule, see why a row number is not a clock after rows are removed, calculate the RMS of two small windows, check which inputs exist at a forecast issue time, and see how overlapping windows share samples.

**Supports.** Lesson 12.2 (section “Define the forecast issue time”) and Lesson 12.3 (sections “Use timestamps rather than filtered row positions”, “Define horizon and information availability” and “Handle overlapping windows and temporal gaps”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

Step 1 uses the numbers of Lesson 12.3: samples every 15 minutes over 24 hours. Steps 2 to 5 use a few invented signal values and times.

All data are synthetic. The signal values are in arbitrary units. They are not measurements from real equipment.

```
Step 3 signal (arbitrary units), 8 samples:
3, -3, 3, -3, 1, 1, 1, 1

Step 4 inputs for a next-day load forecast issued at 18:00:
observed load up to 18:00
published weather forecast, issued at 12:00
tomorrow's measured weather
```


## Step 1. How many 15-minute samples make 24 hours?

Lesson 12.3 says that for complete, regular data sampled every 15 minutes, 96 samples correspond to 24 hours. You can check this with arithmetic.

In [ ]:
# Step 1: samples in 24 hours at one sample every 15 minutes
hours = 24
minutes_per_hour = 60
sample_interval = 15      # minutes between samples
n_samples = hours * minutes_per_hour // sample_interval
print("Samples in 24 hours:", n_samples)

**Expected output**

```
Samples in 24 hours: 96
```

24 × 60 ÷ 15 = 96. This is true only for complete, regular data. The next step shows why.

## Step 2. A row number is not a clock

Here the sensor sends a sample every 15 minutes. The times are stored in minutes since midnight. In the second list, the sample at 45 minutes is missing. The row at position 3 then belongs to a different time.

In [ ]:
# Step 2: row position versus actual timestamp (minutes since midnight)
complete = [0, 15, 30, 45, 60, 75]
with_gap = [0, 15, 30, 60, 75, 90]      # the sample at 45 is missing

print("Row 3, complete data: time", complete[3], "min")
print("Row 3, data with a gap: time", with_gap[3], "min")
print("Time covered by the six rows with a gap:", with_gap[-1] - with_gap[0], "min")
print("Six complete rows would cover:", complete[-1] - complete[0], "min")

**Expected output**

```
Row 3, complete data: time 45 min
Row 3, data with a gap: time 60 min
Time covered by the six rows with a gap: 90 min
Six complete rows would cover: 75 min
```

The same row number points to 45 min in one list and to 60 min in the other. Six rows cover 90 minutes, not 75. Counting rows therefore does not measure time after a gap. Use the actual timestamps, as Lesson 12.3 says. `with_gap[-1]` means the last item of the list.

## Step 3. Calculate the RMS of two windows

Lesson 12.3 says that RMS summarises signal magnitude over a window. To calculate it: square each value, take the mean of the squares, then take the square root. The signal below has two windows of four samples each. The window length is four samples, so it must be recorded together with the sampling interval.

In [ ]:
# Step 3: mean and RMS of two windows (arbitrary units)
signal = [3, -3, 3, -3, 1, 1, 1, 1]
window_length = 4         # samples per window

for start in [0, 4]:
    window = signal[start:start + window_length]
    mean_value = sum(window) / len(window)
    squares = 0
    for value in window:
        squares += value ** 2
    rms = (squares / len(window)) ** 0.5
    print(f"Window starting at sample {start}: mean {mean_value:.1f}, RMS {rms:.1f}")

**Expected output**

```
Window starting at sample 0: mean 0.0, RMS 3.0
Window starting at sample 4: mean 1.0, RMS 1.0
```

The first window swings between +3 and −3. Its mean is 0.0, but its RMS is 3.0. The mean hides the swing and the RMS shows it. The second window is steady at 1, so mean and RMS are both 1.0. An RMS value is only meaningful together with its window length and sampling rate.

## Step 4. Check what exists at the issue time

Lesson 12.3 gives a next-day load forecast issued at 18:00. A feature may be used only if it already exists at that time. The list below marks each input as available or not. The inputs follow the lesson.

In [ ]:
# Step 4: inputs for a next-day load forecast issued at 18:00
inputs = [
    ("observed load up to 18:00", True),
    ("published weather forecast, issued at 12:00", True),
    ("tomorrow's measured weather", False),
]

for name, available in inputs:
    if available:
        print("Use:     ", name)
    else:
        print("Exclude: ", name)

**Expected output**

```
Use:      observed load up to 18:00
Use:      published weather forecast, issued at 12:00
Exclude:  tomorrow's measured weather
```

Observed load before the issue time exists. A forecast issued earlier also exists, but its uncertainty must be recorded. Tomorrow’s measured weather does not exist at 18:00 today. Using it would evaluate a different information scenario. The code only repeats the lesson’s judgement. You must still decide the availability yourself for each real input.

## Step 5. Overlapping windows share samples

Lesson 12.3 says that adjacent windows can share raw samples. Here you list the sample positions of each window. The windows are four samples long and start two samples apart. Then you check which windows share a sample with window 0.

In [ ]:
# Step 5: sample positions of overlapping windows
n_signal = 10             # samples in the signal
window_length = 4
step = 2                  # samples between window starts

windows = []
for start in range(0, n_signal - window_length + 1, step):
    windows.append(list(range(start, start + window_length)))

for number, w in enumerate(windows):
    print(f"Window {number}: samples {w}")

for number, w in enumerate(windows):
    shared = 0
    for sample in w:
        if sample in windows[0]:
            shared += 1
    print(f"Window {number} shares {shared} samples with window 0")

**Expected output**

```
Window 0: samples [0, 1, 2, 3]
Window 1: samples [2, 3, 4, 5]
Window 2: samples [4, 5, 6, 7]
Window 3: samples [6, 7, 8, 9]
Window 0 shares 4 samples with window 0
Window 1 shares 2 samples with window 0
Window 2 shares 0 samples with window 0
Window 3 shares 0 samples with window 0
```

Window 1 shares two samples with window 0. If window 0 were in the training set and window 1 in the test set, the test window would contain training information. Window 2 and later share nothing with window 0. Use a chronological split with a suitable gap, as Lesson 12.3 says. `range(0, 7, 2)` counts 0, 2, 4, 6. `list(...)` turns it into a list.

## Try it

Change a value and observe. In Step 5, change `step = 2` to `step = 4`. Then run the cell again.

**Expected result**

```
Window 0: samples [0, 1, 2, 3]
Window 1: samples [4, 5, 6, 7]
Window 0 shares 4 samples with window 0
Window 1 shares 0 samples with window 0
```

When the step equals the window length, the windows no longer share samples. Only two windows fit into ten samples, and the last two samples are not used. Neighbouring windows can still be correlated in time. Non-overlap does not by itself prove independence.

## Limits

- The signals are tiny, synthetic and in arbitrary units. They do not show how a real sensor behaves.
- Step 2 shows one missing sample. Lesson 12.2 asks you to inspect gaps and choose a treatment that fits the gap. This code does not fill or remove any gap.
- Step 4 repeats a judgement from the lesson. Code cannot decide by itself what is available at the issue time.
- The code does not build a forecast model or an RNN. It only checks windows and information boundaries.

No course lab uses time windows. Lab 2 (Module 7) uses a train and test split on independent designs. This walkthrough shows the timing checks needed for signals.